# RES-142 — Qwen3-Reranker-0.6B Colab setup smoke (G4/A100 only)

**Preparation artifact; all cells are unexecuted and GPU execution is not approved.** This notebook contains only B1 setup and a two-pair synthetic smoke. It has no dataset input path and no pilot/full-benchmark code.

Browser-only account inspection is a separate gate: `APPROVE COLAB INSPECTION` permits viewing available GPU choices, the displayed CU rate, and budget controls only. It does not permit connecting to a runtime or running a cell. GPU setup later requires a separate `APPROVE COLAB SETUP` naming the selected G4 or A100 SKU and an explicit maximum CU charge. No such approval has been given.

The vLLM listener must remain on `127.0.0.1:8000` inside the Colab VM. No tunnel, public port, remote endpoint, dataset upload, Drive mount, or fallback model is used here. B1 ends with a downloadable synthetic evidence archive and a hard review stop.

## 1. Operator parameters — closed by default

This notebook must not be connected to a GPU runtime until setup approval names a SKU and CU cap. Set these fields only after the separate browser inspection and matching explicit approval. `TARGET_GPU` asserts the manually selected hardware; it never provisions or switches GPUs.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

# These remain blank until browser-only inspection and a separate GPU setup approval.
SETUP_AUTHORIZATION = ""
PHASE = "B1_SYNTHETIC_SMOKE"
TARGET_GPU = ""  # Set only to the explicitly approved, UI-confirmed SKU.
OBSERVED_COMPUTE_UNITS_PER_HOUR = None  # Copy the rate shown in the account UI.
APPROVED_MAX_COMPUTE_UNITS = None  # Explicit user-approved maximum for B1.
ALLOW_DATA_TRANSFER = False
PERSIST_TO_DRIVE = False
MAX_RUNTIME_SECONDS = 45 * 60

if SETUP_AUTHORIZATION != "APPROVE COLAB SETUP":
    raise RuntimeError("B1 is locked. Obtain explicit APPROVE COLAB SETUP first.")
if PHASE != "B1_SYNTHETIC_SMOKE" or ALLOW_DATA_TRANSFER or PERSIST_TO_DRIVE:
    raise RuntimeError("This notebook only permits the isolated synthetic B1 smoke.")
if TARGET_GPU not in {"G4", "A100"}:
    raise RuntimeError("Only an explicitly approved, manually selected G4 or A100 is allowed.")
if isinstance(OBSERVED_COMPUTE_UNITS_PER_HOUR, bool) or not isinstance(OBSERVED_COMPUTE_UNITS_PER_HOUR, (int, float)) or OBSERVED_COMPUTE_UNITS_PER_HOUR <= 0:
    raise RuntimeError("Enter the positive CU/hour rate observed in the account UI.")
if isinstance(APPROVED_MAX_COMPUTE_UNITS, bool) or not isinstance(APPROVED_MAX_COMPUTE_UNITS, (int, float)) or APPROVED_MAX_COMPUTE_UNITS <= 0:
    raise RuntimeError("An explicit positive maximum CU charge is required before setup.")
EXPECTED_B1_COMPUTE_UNITS = OBSERVED_COMPUTE_UNITS_PER_HOUR * MAX_RUNTIME_SECONDS / 3600
if EXPECTED_B1_COMPUTE_UNITS > APPROVED_MAX_COMPUTE_UNITS:
    raise RuntimeError("The 45-minute B1 ceiling exceeds the approved CU cap. Do not allocate.")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("This pinned vLLM profile requires Linux x86_64. Stop.")
GLIBC_NAME, GLIBC_VERSION = platform.libc_ver()
if GLIBC_NAME != "glibc" or tuple(int(part) for part in GLIBC_VERSION.split(".")[:2]) < (2, 31):
    raise RuntimeError(f"The vLLM 0.31.0 dependency wheel floor is glibc 2.31; found {GLIBC_NAME} {GLIBC_VERSION}. Stop.")

MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
MODEL_SHA = "e61197ed45024b0ed8a2d74b80b4d909f1255473"
TOKENIZER_REVISION = MODEL_SHA
VLLM_VERSION = "0.31.0"
MAX_MODEL_LEN = 8192
CLIENT_BATCH_SIZE = 8
MAX_NUM_SEQS = 8
MAX_NUM_BATCHED_TOKENS = 16384
GPU_MEMORY_UTILIZATION = 0.50
SERVER_STARTUP_TIMEOUT_SECONDS = 15 * 60
SCORE_TIMEOUT_SECONDS = 120
RUN_STARTED = time.monotonic()
ROOT = Path("/content/res142")
if ROOT.exists():
    raise RuntimeError("Refusing to reuse or overwrite an earlier /content/res142 directory. Start with a fresh runtime.")
ROOT.mkdir(parents=True)
TOOLS_DIR = ROOT / "uv-tools"
UV_BIN = TOOLS_DIR / "bin" / "uv"
VENV = ROOT / "venv"
VENV_PYTHON = VENV / "bin" / "python"
VLLM_BIN = VENV / "bin" / "vllm"
OUTPUT_DIR = ROOT / "b1-synthetic-smoke"
if OUTPUT_DIR.exists():
    raise RuntimeError("Refusing to overwrite an earlier B1 evidence directory.")
if VENV.exists():
    raise RuntimeError("Refusing to reuse an earlier or partial virtual environment. Start with a fresh runtime.")
OUTPUT_DIR.mkdir(parents=True)
def require_time_budget() -> None:
    elapsed = time.monotonic() - RUN_STARTED
    if elapsed > MAX_RUNTIME_SECONDS:
        raise RuntimeError("B1 reached the 45-minute ceiling. Stop and report; do not retry.")
    estimated_cu = OBSERVED_COMPUTE_UNITS_PER_HOUR * elapsed / 3600
    if estimated_cu > APPROVED_MAX_COMPUTE_UNITS:
        raise RuntimeError("B1 reached the approved estimated CU cap. Stop and report.")
print(json.dumps({"authorization": "B1 only", "kernel_python": sys.version.split()[0], "target_gpu": TARGET_GPU, "observed_cu_per_hour": OBSERVED_COMPUTE_UNITS_PER_HOUR, "approved_max_cu": APPROVED_MAX_COMPUTE_UNITS, "estimated_b1_cu_ceiling": EXPECTED_B1_COMPUTE_UNITS}, indent=2))

## 2. Read-only runtime and accelerator check

This checks the accelerator already selected by the operator. It will refuse an unexpected SKU, count, VRAM size, compute capability, or Linux driver. No packages or weights are fetched in this cell.

In [ ]:
def run_readonly(command: list[str]) -> str:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Read-only probe failed: {command[0]} ({result.returncode}).")
    return result.stdout.strip()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

gpu_rows = run_readonly([
    "nvidia-smi", "--query-gpu=name,compute_cap,memory.total,driver_version",
    "--format=csv,noheader,nounits",
]).splitlines()
if len(gpu_rows) != 1:
    raise RuntimeError(f"Expected exactly one visible GPU; found {len(gpu_rows)}. Stop.")
fields = [field.strip() for field in gpu_rows[0].split(",")]
if len(fields) != 4:
    raise RuntimeError("Unexpected nvidia-smi identity row. Stop.")
gpu_name, capability_text, memory_mib_text, driver_text = fields
capability = tuple(int(part) for part in capability_text.split("."))
memory_mib = int(memory_mib_text)
driver = tuple(int(part) for part in driver_text.split("."))
if driver < (575, 57, 8):
    raise RuntimeError(f"Driver {driver_text} is below the proposed CUDA 12.9 floor. Stop.")
if TARGET_GPU == "G4":
    if "RTX PRO 6000" not in gpu_name or "Blackwell" not in gpu_name:
        raise RuntimeError(f"Requested G4 but observed {gpu_name!r}. No automatic fallback.")
    if capability != (12, 0) or memory_mib < 90000:
        raise RuntimeError(f"G4 identity/96 GB floor mismatch: {capability_text}, {memory_mib} MiB.")
else:
    if "A100" not in gpu_name or capability != (8, 0):
        raise RuntimeError(f"Requested A100 but observed {gpu_name!r}. No automatic fallback.")
    if not (38000 <= memory_mib <= 82000):
        raise RuntimeError(f"A100 memory is not a recognized 40/80 GB SKU: {memory_mib} MiB.")
GPU = {"requested_target": TARGET_GPU, "name": gpu_name, "compute_capability": list(capability), "memory_mib": memory_mib, "memory_bytes_reported": memory_mib * 1024 * 1024, "driver_version": driver_text, "observed_compute_units_per_hour": OBSERVED_COMPUTE_UNITS_PER_HOUR, "approved_max_compute_units": APPROVED_MAX_COMPUTE_UNITS}
GPU_MEMORY_OBSERVATIONS = []
def gpu_memory_snapshot() -> dict[str, int]:
    row = run_readonly(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"])
    used_mib, total_mib, utilization_pct = (int(part.strip()) for part in row.split(","))
    if used_mib * 100 > total_mib * 80 or (total_mib - used_mib) * 100 < total_mib * 20:
        raise RuntimeError(f"GPU memory stop threshold exceeded ({used_mib}/{total_mib} MiB).")
    snapshot = {"used_mib": used_mib, "total_mib": total_mib, "utilization_pct": utilization_pct}
    GPU_MEMORY_OBSERVATIONS.append(snapshot)
    return snapshot
gpu_memory_snapshot()
print(json.dumps(GPU, indent=2))

## 3. Bootstrap uv, managed Python 3.12, and the hash-locked environment

After a later setup approval, this cell accepts the reviewed lock and template files, verifies their SHA-256 values, and installs the exact hash-locked graph into a separate Python 3.12 venv. The current Jupyter kernel may use another Python version; it only orchestrates subprocesses. All vLLM, torch, probes, and scoring commands run through the venv interpreter.

In [ ]:
from google.colab import files
EXPECTED_LOCK_SHA256 = "d8a87ad2f1f2a7faccfb96fee9fa859995f61caa7b8a7b845e268888c4c111dc"
EXPECTED_TEMPLATE_SHA256 = "bf7b104801a4e4ad6a147d732e2d5a837a36f708cdbd1b871b864979d3e10acd"
uploaded = files.upload()
EXPECTED_UPLOADS = {"res142-colab.lock.txt", "res142_qwen3_reranker.jinja"}
if set(uploaded) != EXPECTED_UPLOADS:
    raise RuntimeError(f"Upload exactly these reviewed files: {sorted(EXPECTED_UPLOADS)}")
LOCK_PATH = ROOT / "res142-colab.lock.txt"
TEMPLATE_PATH = ROOT / "qwen3_reranker.jinja"
LOCK_PATH.write_bytes(uploaded["res142-colab.lock.txt"])
TEMPLATE_PATH.write_bytes(uploaded["res142_qwen3_reranker.jinja"])
if sha256_file(LOCK_PATH) != EXPECTED_LOCK_SHA256:
    raise RuntimeError("Candidate lock digest mismatch; stop before dependency setup.")
if sha256_file(TEMPLATE_PATH) != EXPECTED_TEMPLATE_SHA256:
    raise RuntimeError("Qwen3 template digest mismatch; stop before dependency setup.")
UV_VERSION = "0.12.4"
if TOOLS_DIR.exists() or (ROOT / "managed-python").exists():
    raise RuntimeError("Refusing to reuse partial uv/Python bootstrap state. Start with a fresh runtime.")
TOOLS_DIR.mkdir(parents=True)
bootstrap_command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-deps", "--only-binary=:all:", "--target", str(TOOLS_DIR), f"uv=={UV_VERSION}"]
bootstrap = subprocess.run(bootstrap_command, capture_output=True, text=True, check=False, timeout=300)
if bootstrap.returncode != 0 or not UV_BIN.is_file():
    raise RuntimeError("Pinned uv bootstrap failed; stop without changing versions.")
if run_readonly([str(UV_BIN), "--version"]) != f"uv {UV_VERSION}":
    raise RuntimeError("uv bootstrap version mismatch.")
uv_env = os.environ.copy()
uv_env["UV_PYTHON_INSTALL_DIR"] = str(ROOT / "managed-python")
uv_env["UV_PYTHON_PREFERENCE"] = "only-managed"
python_install = subprocess.run([str(UV_BIN), "python", "install", "3.12"], env=uv_env, capture_output=True, text=True, check=False, timeout=900)
require_time_budget()
if python_install.returncode != 0:
    raise RuntimeError("Could not provision managed Python 3.12; stop.")
venv_create = subprocess.run([str(UV_BIN), "venv", "--python", "3.12", str(VENV)], env=uv_env, capture_output=True, text=True, check=False, timeout=300)
require_time_budget()
if venv_create.returncode != 0:
    raise RuntimeError("Could not create the isolated Python 3.12 venv; stop.")
install_started = time.perf_counter()
install_command = [str(UV_BIN), "pip", "sync", "--python", str(VENV_PYTHON), "--torch-backend", "cu129", "--only-binary", ":all:", "--require-hashes", "--strict", str(LOCK_PATH)]
install = subprocess.run(install_command, env=uv_env, capture_output=True, text=True, check=False, timeout=1800)
require_time_budget()
if install.returncode != 0:
    raise RuntimeError("Hash-locked binary-wheel sync failed; stop without changing pins or GPU.")
install_seconds = time.perf_counter() - install_started
pip_check = subprocess.run([str(UV_BIN), "pip", "check", "--python", str(VENV_PYTHON)], env=uv_env, capture_output=True, text=True, check=False)
if pip_check.returncode != 0:
    raise RuntimeError("Dependency check failed; do not smoke or resolve by upgrading packages.")
pip_freeze = run_readonly([str(UV_BIN), "pip", "freeze", "--python", str(VENV_PYTHON)])
PIP_VERSION = run_readonly([str(VENV_PYTHON), "--version"])
(OUTPUT_DIR / "pip-freeze.txt").write_text(pip_freeze + "\n", encoding="utf-8")
print(f"Install seconds: {install_seconds:.3f}; kernel Python: {sys.version.split()[0]}; isolated interpreter: {PIP_VERSION}; full package freeze captured.")
print(pip_check.stdout.strip() or "dependency check: no broken requirements")

## 4. Validate CUDA/PyTorch, exact package versions, and BF16 support

No model weights have been loaded yet. A CUDA/PyTorch mismatch or any package drift is a hard stop.

In [ ]:
runtime_probe = r'''import importlib.metadata as md, json, platform, sys, torch
names = ["vllm", "numba", "torch", "torchvision", "torchaudio", "transformers", "huggingface-hub", "tokenizers", "safetensors"]
versions = {name: md.version(name) for name in names}
props = torch.cuda.get_device_properties(0)
print(json.dumps({"python": sys.version.split()[0], "platform": platform.platform(), "glibc": platform.libc_ver(), "packages": versions, "cuda_runtime": torch.version.cuda, "gpu_name": torch.cuda.get_device_name(0), "compute_capability": list(torch.cuda.get_device_capability(0)), "gpu_total_memory_bytes": props.total_memory, "bf16_supported": torch.cuda.is_bf16_supported()}))
'''probe = subprocess.run([str(VENV_PYTHON), "-c", runtime_probe], capture_output=True, text=True, check=False)
if probe.returncode != 0:
    raise RuntimeError("PyTorch/CUDA probe failed; stop.")
SOFTWARE = json.loads(probe.stdout.strip())
expected_versions = {"vllm": "0.31.0", "numba": "0.65.0", "torchvision": "0.28.0", "torchaudio": "2.11.0", "transformers": "5.10.4", "huggingface-hub": "1.31.0", "tokenizers": "0.22.1", "safetensors": "0.6.2"}
for package, expected in expected_versions.items():
    if SOFTWARE["packages"][package].split("+", 1)[0] != expected:
        raise RuntimeError(f"Pinned package drift: {package}={SOFTWARE['packages'][package]}, expected {expected}.")
if not SOFTWARE["packages"]["torch"].startswith("2.13.0") or not str(SOFTWARE["cuda_runtime"]).startswith("12.9"):
    raise RuntimeError("Expected PyTorch 2.13.0 with CUDA 12.9. Stop.")
if not SOFTWARE["bf16_supported"]:
    raise RuntimeError("The selected GPU/runtime does not report BF16 support. Stop; do not switch dtype.")
if SOFTWARE["gpu_name"] != GPU["name"] or SOFTWARE["compute_capability"] != GPU["compute_capability"]:
    raise RuntimeError("GPU identity changed between nvidia-smi and PyTorch. Stop.")
print(json.dumps(SOFTWARE, indent=2))

## 5. Verify the pinned template and start the private scoring server

The exact Jinja bytes are supplied as a reviewed file and reproduced below from the pinned source. Their digest is verified before launch. The server is bound to loopback only. It loads the model at the immutable model/tokenizer commit; this is the first cell that downloads model weights.

In [ ]:
SCORE_TEMPLATE = "\n".join([
    '<|im_start|>system',
    'Judge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>',
    '<|im_start|>user',
    '<Instruct>: {{ instruction | default(instruct | default(messages | selectattr("role", "eq", "system") | map(attribute="content") | first | default("Given a web search query, retrieve relevant passages that answer the query", true), true), true) }}',
    '<Query>: {{ messages | selectattr("role", "eq", "query") | map(attribute="content") | first }}',
    '<Document>: {{ messages | selectattr("role", "eq", "document") | map(attribute="content") | first }}<|im_end|>',
    '<|im_start|>assistant',
    '<think>',
    '',
    '</think>',
    '',
    '',
])
TEMPLATE_PATH = ROOT / "qwen3_reranker.jinja"
TEMPLATE_PATH.write_text(SCORE_TEMPLATE, encoding="utf-8", newline="\n")
TEMPLATE_SHA256 = hashlib.sha256(TEMPLATE_PATH.read_bytes()).hexdigest()
EXPECTED_TEMPLATE_SHA256 = "bf7b104801a4e4ad6a147d732e2d5a837a36f708cdbd1b871b864979d3e10acd"
if TEMPLATE_SHA256 != EXPECTED_TEMPLATE_SHA256:
    raise RuntimeError(f"Qwen3 score template digest mismatch: {TEMPLATE_SHA256}. Stop.")

TASK_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
HF_HOME = ROOT / "hf-home"
HF_HOME.mkdir(exist_ok=True)
server_args = [
    str(VLLM_BIN), "serve", MODEL_ID,
    "--revision", MODEL_SHA, "--tokenizer-revision", TOKENIZER_REVISION,
    "--runner", "pooling", "--dtype", "bfloat16",
    "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--tensor-parallel-size", "1", "--seed", "0",
    "--hf_overrides", json.dumps({
        "architectures": ["Qwen3ForSequenceClassification"],
        "classifier_from_token": ["no", "yes"],
        "is_original_qwen3_reranker": True,
    }, separators=(",", ":")),
    "--chat-template", str(TEMPLATE_PATH),
    "--host", "127.0.0.1", "--port", "8000",
]
server_env = os.environ.copy()
server_env["HF_HOME"] = str(HF_HOME)
server_env["HF_HUB_DISABLE_TELEMETRY"] = "1"
SERVER_LOG = OUTPUT_DIR / "vllm.log"
server_log_handle = SERVER_LOG.open("w", encoding="utf-8")
print(json.dumps({"argv": server_args, "template_sha256": TEMPLATE_SHA256, "bind": "127.0.0.1:8000"}, indent=2))

## 6. Synthetic scoring, evidence, and cleanup

Exactly two synthetic query-document pairs are scored. The code validates response shape and numeric values only; two pairs are not evidence of scientific quality. The process is terminated in `finally`, and the only export is a small synthetic evidence archive.

In [ ]:
import math
import urllib.error
import urllib.request

synthetic_query = "Which planet is called the Red Planet?"
synthetic_documents = ["Mars is called the Red Planet.", "Venus is Earth's twin."]
if len(synthetic_documents) != 2:
    raise RuntimeError("B1 is restricted to exactly two synthetic pairs.")
synthetic_input = {"query": synthetic_query, "documents": synthetic_documents}
input_sha256 = hashlib.sha256(json.dumps(synthetic_input, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
process = None
started = time.monotonic()
startup_seconds = None
smoke_ms = None
scores = []
try:
    process = subprocess.Popen(server_args, stdout=server_log_handle, stderr=subprocess.STDOUT, env=server_env)
    deadline = time.monotonic() + SERVER_STARTUP_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        require_time_budget()
        gpu_memory_snapshot()
        if process.poll() is not None:
            raise RuntimeError("vLLM exited during startup. Stop; do not retry or change the profile.")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as response:
                if response.status == 200:
                    startup_seconds = time.monotonic() - started
                    break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            time.sleep(2)
    if startup_seconds is None:
        raise RuntimeError("vLLM did not become healthy within 15 minutes. Stop.")
    model_snapshot = HF_HOME / "hub" / "models--Qwen--Qwen3-Reranker-0.6B" / "snapshots" / MODEL_SHA
    required_model_files = {"config.json", "tokenizer.json", "tokenizer_config.json", "model.safetensors"}
    if not model_snapshot.is_dir() or not required_model_files.issubset({path.name for path in model_snapshot.iterdir()}):
        raise RuntimeError("The local Hugging Face cache does not prove the pinned model snapshot/files. Stop.")
    hash_started = time.perf_counter()
    model_snapshot_files = [{"name": path.name, "size_bytes": path.stat().st_size, "sha256": sha256_file(path)} for path in sorted(model_snapshot.iterdir()) if path.is_file()]
    model_file_hashing_seconds = time.perf_counter() - hash_started
    require_time_budget()
    gpu_memory_snapshot()
    require_time_budget()
    payload = {
        "model": MODEL_ID,
        "queries": synthetic_query,
        "documents": synthetic_documents,
        "instruction": TASK_INSTRUCTION,
        "use_activation": True,
        "max_tokens_per_query": 1024,
        "max_tokens_per_doc": 6144,
    }
    request = urllib.request.Request("http://127.0.0.1:8000/score", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json", "Accept": "application/json"}, method="POST")
    score_started = time.perf_counter()
    with urllib.request.urlopen(request, timeout=SCORE_TIMEOUT_SECONDS) as response:
        response_payload = json.loads(response.read().decode("utf-8"))
    smoke_ms = (time.perf_counter() - score_started) * 1000.0
    if process.poll() is not None:
        raise RuntimeError("vLLM exited after scoring; stop and report.")
    rows = response_payload.get("data")
    if response_payload.get("model") != MODEL_ID:
        raise RuntimeError("Scoring response model identity differs from the pinned model ID.")
    if not isinstance(rows, list) or len(rows) != 2:
        raise RuntimeError("Scoring response cardinality is not exactly two.")
    for row in rows:
        if not isinstance(row, dict) or isinstance(row.get("index"), bool) or row.get("index") not in {0, 1}:
            raise RuntimeError("Scoring response has an invalid/foreign index.")
        value = row.get("score")
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(float(value)):
            raise RuntimeError("Scoring response has a non-finite score.")
        scores.append({"synthetic_candidate_position": row["index"], "score": float(value)})
    if {row["synthetic_candidate_position"] for row in scores} != {0, 1}:
        raise RuntimeError("Scoring response indices are not a bijection of the two inputs.")
    gpu_memory_snapshot()
    require_time_budget()
finally:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
    server_log_handle.close()
    if process is not None and process.poll() is None:
        raise RuntimeError("vLLM process cleanup failed.")

def host_memory_snapshot() -> dict[str, int]:
    values = {}
    with Path("/proc/meminfo").open(encoding="utf-8") as handle:
        for line in handle:
            key, raw, *_ = line.split()
            if key in {"MemTotal:", "MemAvailable:"}:
                values[key[:-1]] = int(raw) * 1024
    return values

scores_path = OUTPUT_DIR / "synthetic-scores.json"
scores_bytes = json.dumps(scores, sort_keys=True, separators=(",", ":")).encode("utf-8") + b"\n"
scores_path.write_bytes(scores_bytes)
template_bytes = TEMPLATE_PATH.read_bytes()
shutil.copy2(LOCK_PATH, OUTPUT_DIR / "res142-colab.lock.txt")
artifact_files = {
    name: {"size_bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for name, path in sorted({
        "pip-freeze.txt": OUTPUT_DIR / "pip-freeze.txt",
        "synthetic-scores.json": scores_path,
        "vllm.log": SERVER_LOG,
        "qwen3_reranker.jinja": TEMPLATE_PATH,
        "res142-colab.lock.txt": OUTPUT_DIR / "res142-colab.lock.txt",
    }.items())
}
manifest = {
    "manifest_revision": "res142-colab-b1-smoke-v1",
    "phase": PHASE,
    "stop_status": "b1-synthetic-smoke-completed",
    "preparation_base_git_sha": "67edcc785af2084e2cc4d8c27b08bea43a5f4bd3",
    "res142_implementation_git_sha": None,
    "evaluation_code_sha": None,
    "model": {"repo": MODEL_ID, "model_sha": MODEL_SHA, "tokenizer_revision": TOKENIZER_REVISION, "snapshot_commit_confirmed": model_snapshot.name, "snapshot_files": model_snapshot_files},
    "vllm_wheel_sha256": "5176ab746f4e736c4a354a11d0620abf0a941de287cad937ac34039293ca6f5f",
    "python": SOFTWARE["python"],
    "packages": SOFTWARE["packages"],
    "venv_python": PIP_VERSION,
    "uv_version": UV_VERSION,
    "pip_freeze_sha256": sha256_file(OUTPUT_DIR / "pip-freeze.txt"),
    "pip_check": pip_check.stdout.strip() or "no broken requirements",
    "gpu": GPU,
    "compute_units": {"observed_rate_per_hour": OBSERVED_COMPUTE_UNITS_PER_HOUR, "approved_maximum": APPROVED_MAX_COMPUTE_UNITS, "estimated_b1_ceiling": EXPECTED_B1_COMPUTE_UNITS},
    "cuda_runtime": SOFTWARE["cuda_runtime"],
    "serving": {"argv": server_args, "endpoint": "http://127.0.0.1:8000/score", "template_sha256": hashlib.sha256(template_bytes).hexdigest()},
    "prompt": {"instruction": TASK_INSTRUCTION, "instruction_sha256": hashlib.sha256(TASK_INSTRUCTION.encode("utf-8")).hexdigest()},
    "inputs": {"kind": "synthetic-only", "pair_count": 2, "input_sha256": input_sha256, "dataset_sha256": None, "candidate_snapshot_sha256": None, "baseline_run_sha256": None},
    "results": {"score_count": len(scores), "scores_sha256": sha256_file(scores_path)},
    "timing": {"install_seconds": install_seconds, "server_startup_seconds": startup_seconds, "model_file_hashing_seconds": model_file_hashing_seconds, "smoke_request_ms": smoke_ms, "p50_ms": None, "p95_ms": None, "p99_ms": None},
    "cleanup": {"server_terminated": process is not None and process.poll() is not None, "server_returncode": process.returncode if process is not None else None},
    "resources": {"gpu_memory_observations": GPU_MEMORY_OBSERVATIONS, "gpu_memory_high_water_mib": max(row["used_mib"] for row in GPU_MEMORY_OBSERVATIONS), "host_memory": host_memory_snapshot(), "child_process_max_rss_kib": __import__("resource").getrusage(__import__("resource").RUSAGE_CHILDREN).ru_maxrss},
    "artifact_files": artifact_files,
    "runtime_seconds": time.monotonic() - RUN_STARTED,
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_bytes(json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode("utf-8") + b"\n")
(OUTPUT_DIR / "manifest.sha256").write_text(sha256_file(manifest_path) + "  manifest.json\n", encoding="utf-8")
archive_path = Path(shutil.make_archive(str(ROOT / "res142-b1-synthetic-smoke"), "zip", root_dir=OUTPUT_DIR))
archive_sha256 = sha256_file(archive_path)
print(json.dumps(manifest, indent=2, sort_keys=True))
print(json.dumps({"archive": str(archive_path), "archive_sha256": archive_sha256}, indent=2))
from google.colab import files
files.download(str(archive_path))
print("B1 complete: verify archive hashes locally, then STOP for review.")

## 7. Persist the B1 smoke artifact and stop

B1 uses no Drive and writes no scientific or user data. Download the generated synthetic evidence files through the Colab UI, verify the manifest and all hashes on the Windows workstation, then use the Colab UI to disconnect and delete the runtime so GPU allocation ends. The Colab filesystem, venv, uv-managed Python, and HF model cache remain until the runtime is deleted.

**Hard stop:** after B1, report actual hardware, CU rate/cap, package freeze, model identity, command, smoke response shape, timings, resource readings, and artifact hashes. Do not add candidate inputs or change `PHASE`. The future B2 pilot is a uniform deterministic 20-query SciFact sample (maximum 1,000 pairs), not the full 300-query split. It requires separate `APPROVE RES142 PILOT`, an approved CU cap, and a frozen, rights-qualified candidate bundle. Full 300-query evaluation requires `APPROVE RES142 FULL BENCHMARK`.